In [1]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

In [3]:
PROJECT_ROOT = Path.cwd().resolve()

# If the notebook is inside CarePulse/notebooks,
# move one level up to CarePulse
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

C:\Users\shiva\OneDrive\Desktop\CarePulse


In [4]:
INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "cxr"
    / "cxr_labeled.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "cxr"
    / "eda"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [6]:
df = pd.read_csv(INPUT_FILE)

print("Dataset shape:", df.shape)

Dataset shape: (9834, 18)


In [7]:
FINDINGS = [
    "pneumonia",
    "pleural_effusion",
    "pneumothorax",
    "atelectasis",
    "consolidation",
    "pulmonary_edema",
    "cardiomegaly",
    "opacity",
]

rows = []

for finding in FINDINGS:
    col = f"label_{finding}"

    positive = (df[col] == 1).sum()
    negative = (df[col] == 0).sum()
    uncertain = (df[col] == -1).sum()
    not_mentioned = df[col].isna().sum()

    rows.append({
        "Finding": finding,
        "Positive": positive,
        "Negative": negative,
        "Uncertain": uncertain,
        "Not Mentioned": not_mentioned,
        "Usable": positive + negative,
    })

distribution_df = pd.DataFrame(rows)

distribution_df

,Finding,Positive,Negative,Uncertain,Not Mentioned,Usable
0,pneumonia,804,803,195,8032,1607
1,pleural_effusion,3499,2601,239,3495,6100
2,pneumothorax,485,5050,42,4257,5535
3,atelectasis,3137,145,288,6264,3282
4,consolidation,688,1004,105,8037,1692
5,pulmonary_edema,1525,1119,90,7100,2644
6,cardiomegaly,1647,101,18,8068,1748
7,opacity,2998,321,348,6167,3319


In [8]:
split_rows = []

for split in ["Train", "Test"]:
    split_df = df[df["split"] == split]

    for finding in FINDINGS:
        col = f"label_{finding}"

        split_rows.append({
            "Split": split,
            "Finding": finding,
            "Positive": (split_df[col] == 1).sum(),
            "Negative": (split_df[col] == 0).sum(),
            "Uncertain": (split_df[col] == -1).sum(),
        })

split_df_summary = pd.DataFrame(split_rows)

split_df_summary

,Split,Finding,Positive,Negative,Uncertain
0,Train,pneumonia,676,689,170
1,Train,pleural_effusion,2897,2256,192
2,Train,pneumothorax,447,4245,34
3,Train,atelectasis,2658,129,233
4,Train,consolidation,585,872,82
5,Train,pulmonary_edema,1196,987,74
6,Train,cardiomegaly,1314,87,17
7,Train,opacity,2474,252,285
8,Test,pneumonia,128,114,25
9,Test,pleural_effusion,602,345,47


In [9]:
train_df = df[df["split"] == "Train"]

for finding in FINDINGS:

    col = f"label_{finding}"

    print("\n" + "=" * 70)
    print(f"FINDING: {finding.upper()}")
    print("=" * 70)

    print("\n--- POSITIVE ---")

    samples = train_df[train_df[col] == 1].sample(
        min(3, (train_df[col] == 1).sum()),
        random_state=42
    )

    for _, row in samples.iterrows():
        print(f"\nImage: {row['image_id']}")
        print(row["report"])

    print("\n--- NEGATIVE ---")

    samples = train_df[train_df[col] == 0].sample(
        min(3, (train_df[col] == 0).sum()),
        random_state=42
    )

    for _, row in samples.iterrows():
        print(f"\nImage: {row['image_id']}")
        print(row["report"])


FINDING: PNEUMONIA

--- POSITIVE ---

Image: s50879011
In comparison with the study of ___, there is little interval change. Continued enlargement of the cardiac silhouette with central venous line extending to the right atrium. Areas of increased opacification at the right base could reflect atelectasis and small effusion, though in view of the clinical history, supervening pneumonia must be seriously considered. No definite pulmonary edema.

Image: s50399925
In comparison with the study of ___, the nasogastric tube again extends to the distal stomach. Cardiomegaly persists and there is now a bilateral opacification, slightly more prominent on the right, which most likely represents developing pulmonary vascular congestion. In the appropriate clinical setting, superimposed pneumonia could be considered.

Image: s50481961
Single AP view of the chest was reviewed. Since the most recent prior study, there has been placement of a right internal jugular line with tip terminating in the mi